# Final model comparison and error analysis

This notebook compares the selected LightGBM baseline with SchNet, tests whether their errors are complementary, and evaluates a validation-selected weighted ensemble. It reuses saved metrics, predictions, checkpoints, and deterministic feature caches; no new architecture is introduced.

## Notebook contract

**Goal:** consolidate the validated model comparison and assess ensemble complementarity.  
**Inputs:** saved classical/SchNet metrics, checkpoints, feature cache, and prediction artifacts.  
**Outputs:** final metrics, aligned random/scaffold predictions, presentation figures, and conclusions.  
**Main questions:** Which approach is best for interpolation, unseen scaffolds, and final delivery?


## 1. Setup and exact split reproduction

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
# Standard library
import json

# Third-party
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.metrics import mean_absolute_error

# Project
from src.analysis import build_prediction_frame, difficult_tables, error_associations
from src.config import (
    CLASSICAL_ANALYSIS_FEATURES, CLASSICAL_METRICS, CLASSICAL_RANDOM_PREDICTIONS,
    CLASSICAL_SCAFFOLD_PREDICTIONS, DATA_PATH, FINAL_FIGURE_DIR,
    FINAL_RANDOM_PREDICTIONS, FINAL_SCAFFOLD_PREDICTIONS,
    FINAL_VALIDATION_PREDICTIONS, PROJECT_ROOT, RANDOM_SEED, RESULTS_DIR,
    SCHNET_METRICS, SCHNET_RANDOM_PREDICTIONS, SCHNET_SCAFFOLD_PREDICTIONS,
    SPLIT_ARTIFACT,
)
from src.data import (
    labeled_molecules, load_dataset, load_required_csv, load_required_numpy,
    validate_analysis_features, validate_prediction_alignment, validate_prediction_split,
    validate_split_indices,
)
from src.evaluation import metric_row
from src.plotting import MODEL_COLORS, configure_matplotlib, save_figure


In [ ]:
ROOT = PROJECT_ROOT
FIGURE_DIR = FINAL_FIGURE_DIR
RANDOM_STATE = RANDOM_SEED
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
configure_matplotlib()

df = load_dataset(DATA_PATH)
labeled = labeled_molecules(df)
y = labeled["gap"].to_numpy(dtype=np.float32)
split_data = load_required_numpy(SPLIT_ARTIFACT, "notebooks/02_classical_baselines.ipynb")
random_train_idx, random_val_idx, random_test_idx = (
    split_data["random_train"], split_data["random_validation"], split_data["random_test"])
scaffold_train_idx, scaffold_val_idx, scaffold_test_idx = (
    split_data["scaffold_train"], split_data["scaffold_validation"], split_data["scaffold_test"])
validate_split_indices(random_train_idx, random_val_idx, random_test_idx, len(labeled))
validate_split_indices(scaffold_train_idx, scaffold_val_idx, scaffold_test_idx, len(labeled))


## 2. Load and validate saved predictions

Notebook 02 supplies classical validation/test predictions and notebook 03 supplies SchNet validation/test predictions. Missing inputs raise an actionable error; this notebook never constructs or trains either model.


In [ ]:
classical_random = load_required_csv(
    CLASSICAL_RANDOM_PREDICTIONS, "notebooks/02_classical_baselines.ipynb")
classical_scaffold = load_required_csv(
    CLASSICAL_SCAFFOLD_PREDICTIONS, "notebooks/02_classical_baselines.ipynb")
schnet_random = load_required_csv(
    SCHNET_RANDOM_PREDICTIONS, "notebooks/03_schnet.ipynb")
schnet_scaffold = load_required_csv(
    SCHNET_SCAFFOLD_PREDICTIONS, "notebooks/03_schnet.ipynb")

for frame in (classical_random, classical_scaffold, schnet_random, schnet_scaffold):
    validate_prediction_alignment(frame, labeled)
for frame, indices, split in [
    (classical_random.query("split == 'random_validation'"), random_val_idx, "random_validation"),
    (classical_random.query("split == 'random_test'"), random_test_idx, "random_test"),
    (classical_scaffold.query("split == 'scaffold_validation'"), scaffold_val_idx, "scaffold_validation"),
    (classical_scaffold.query("split == 'scaffold_test'"), scaffold_test_idx, "scaffold_test"),
]:
    validate_prediction_split(frame, indices, split=split)


### 2.2 Align model predictions and load molecular metadata


In [ ]:
random_validation = classical_random.query("split == 'random_validation'").reset_index(drop=True)
scaffold_validation = classical_scaffold.query("split == 'scaffold_validation'").reset_index(drop=True)
random_test = classical_random.query("split == 'random_test'").reset_index(drop=True)
scaffold_test = classical_scaffold.query("split == 'scaffold_test'").reset_index(drop=True)
random_schnet_validation = schnet_random.query("split == 'random_validation'").reset_index(drop=True)
scaffold_schnet_validation = schnet_scaffold.query("split == 'scaffold_validation'").reset_index(drop=True)
random_schnet_test = schnet_random.query("split == 'random_test'").reset_index(drop=True)
scaffold_schnet_test = schnet_scaffold.query("split == 'scaffold_test'").reset_index(drop=True)
for classical, schnet in [(random_validation, random_schnet_validation),
                          (scaffold_validation, scaffold_schnet_validation),
                          (random_test, random_schnet_test), (scaffold_test, scaffold_schnet_test)]:
    np.testing.assert_array_equal(classical["mol_id"], schnet["mol_id"])
    np.testing.assert_allclose(classical["target"], schnet["target"], rtol=0, atol=0)

random_val_true, random_lgbm_val = random_validation["target"].to_numpy(), random_validation["prediction"].to_numpy()
random_schnet_val = random_schnet_validation["prediction"].to_numpy()
scaffold_val_true, scaffold_lgbm_val = scaffold_validation["target"].to_numpy(), scaffold_validation["prediction"].to_numpy()
scaffold_schnet_val = scaffold_schnet_validation["prediction"].to_numpy()
random_test_true, random_lgbm_test = random_test["target"].to_numpy(), random_test["prediction"].to_numpy()
random_schnet_test = random_schnet_test["prediction"].to_numpy()
scaffold_test_true, scaffold_lgbm_test = scaffold_test["target"].to_numpy(), scaffold_test["prediction"].to_numpy()
scaffold_schnet_test = scaffold_schnet_test["prediction"].to_numpy()

analysis_features = load_required_numpy(
    CLASSICAL_ANALYSIS_FEATURES, "notebooks/02_classical_baselines.ipynb")
validate_analysis_features(analysis_features, labeled)
X_size = analysis_features["X_size"]
X_geometry = analysis_features["X_geometry"]
scaffolds = analysis_features["scaffolds"]
classical_metrics = load_required_csv(CLASSICAL_METRICS, "notebooks/02_classical_baselines.ipynb")
best_classical_random = classical_metrics.query("split_strategy == 'random'").sort_values("validation_mae").iloc[0]
classical_family = best_classical_random["model"]
print("Loaded and aligned classical and SchNet artifacts; no model reconstruction performed.")


## 3. Consolidated model results

In [ ]:
schnet_metrics = load_required_csv(SCHNET_METRICS, "notebooks/03_schnet.ipynb")
base_rows = schnet_metrics.copy()
base_rows["absolute_test_mae_difference_vs_lightgbm"] = np.nan
base_rows["relative_test_mae_difference_vs_lightgbm_pct"] = np.nan
for split_name in ["random", "scaffold"]:
    reference = base_rows[(base_rows["split"] == split_name) & base_rows["model"].str.contains("LightGBM")]["test_mae"].iloc[0]
    mask = base_rows["split"].eq(split_name)
    base_rows.loc[mask, "absolute_test_mae_difference_vs_lightgbm"] = base_rows.loc[mask, "test_mae"] - reference
    base_rows.loc[mask, "relative_test_mae_difference_vs_lightgbm_pct"] = (
        100 * (base_rows.loc[mask, "test_mae"] - reference) / reference
    )
display(base_rows)

wide_summary = base_rows.pivot(index="split", columns="model", values="test_mae")
wide_summary["schnet_minus_lightgbm_absolute"] = (
    wide_summary["SchNet"] - wide_summary[classical_family]
)
wide_summary["schnet_minus_lightgbm_relative_pct"] = (
    100 * wide_summary["schnet_minus_lightgbm_absolute"] / wide_summary[classical_family]
)
display(wide_summary)

## 4. Comparative per-molecule error data

In [ ]:
SIZE_NAMES = [
    "n_atoms", "n_heavy_atoms", "n_bonds", "n_atom_types",
    "element_H", "element_C", "element_N", "element_O", "element_F",
    "bond_type_1", "bond_type_1.5", "bond_type_2", "bond_type_3",
]
GEOMETRY_NAMES = [
    "radius_of_gyration", "mean_bond_length", "std_bond_length",
    "max_atomic_distance", "coordinate_cov_eigenvalue_1",
    "coordinate_cov_eigenvalue_2", "coordinate_cov_eigenvalue_3",
]


random_predictions = build_prediction_frame(
    split="random", train_indices=random_train_idx, test_indices=random_test_idx,
    targets=random_test_true, lightgbm_predictions=random_lgbm_test,
    schnet_predictions=random_schnet_test, molecule_table=labeled, scaffolds=scaffolds,
    size_matrix=X_size, size_names=SIZE_NAMES,
    geometry_matrix=X_geometry, geometry_names=GEOMETRY_NAMES,
)
scaffold_predictions = build_prediction_frame(
    split="scaffold", train_indices=scaffold_train_idx, test_indices=scaffold_test_idx,
    targets=scaffold_test_true, lightgbm_predictions=scaffold_lgbm_test,
    schnet_predictions=scaffold_schnet_test, molecule_table=labeled, scaffolds=scaffolds,
    size_matrix=X_size, size_names=SIZE_NAMES,
    geometry_matrix=X_geometry, geometry_names=GEOMETRY_NAMES,
)

random_predictions.to_csv(RESULTS_DIR / "final_predictions_random_test.csv", index=False)
scaffold_predictions.to_csv(RESULTS_DIR / "final_predictions_scaffold_test.csv", index=False)



display(Markdown("**Random-test error associations**"))
display(error_associations(random_predictions, GEOMETRY_NAMES))
display(Markdown("**Scaffold-test error associations**"))
display(error_associations(scaffold_predictions, GEOMETRY_NAMES))

element_rows = []
for split_name, frame in [("random", random_predictions), ("scaffold", scaffold_predictions)]:
    for element in ["C", "N", "O", "F"]:
        present = frame[f"element_{element}"] > 0
        element_rows.append({
            "split": split_name, "element": element,
            "molecules_with_element": int(present.sum()),
            "mean_schnet_improvement": frame.loc[present, "schnet_improvement"].mean(),
            "median_schnet_improvement": frame.loc[present, "schnet_improvement"].median(),
        })
element_analysis = pd.DataFrame(element_rows)
display(Markdown("**Element-composition summary (positive favors SchNet)**"))
display(element_analysis)

## 5. Difficult molecules

In [ ]:

for split_name, frame in [("random", random_predictions), ("scaffold", scaffold_predictions)]:
    display(Markdown(f"### {split_name.title()} test"))
    for title, table in difficult_tables(frame).items():
        display(Markdown(f"**{title}**"))
        display(table)

display(Markdown(
    "These rankings identify candidates for chemical review; they do not establish a causal "
    "relationship between a substructure and model performance."
))

## 6. Residual complementarity and validation-selected ensemble

In [ ]:
random_lgbm_val_residual = random_val_true - random_lgbm_val
random_schnet_val_residual = random_val_true - random_schnet_val
random_test_lgbm_residual = random_test_true - random_lgbm_test
random_test_schnet_residual = random_test_true - random_schnet_test
scaffold_test_lgbm_residual = scaffold_test_true - scaffold_lgbm_test
scaffold_test_schnet_residual = scaffold_test_true - scaffold_schnet_test

residual_correlations = pd.DataFrame([
    {"subset": "random validation", "pearson": np.corrcoef(random_lgbm_val_residual, random_schnet_val_residual)[0,1],
     "spearman": pd.Series(random_lgbm_val_residual).corr(pd.Series(random_schnet_val_residual), method="spearman")},
    {"subset": "random test", "pearson": np.corrcoef(random_test_lgbm_residual, random_test_schnet_residual)[0,1],
     "spearman": pd.Series(random_test_lgbm_residual).corr(pd.Series(random_test_schnet_residual), method="spearman")},
    {"subset": "scaffold test", "pearson": np.corrcoef(scaffold_test_lgbm_residual, scaffold_test_schnet_residual)[0,1],
     "spearman": pd.Series(scaffold_test_lgbm_residual).corr(pd.Series(scaffold_test_schnet_residual), method="spearman")},
])
display(residual_correlations)

weights = np.linspace(0, 1, 101)
validation_curve = pd.DataFrame({
    "schnet_weight": weights,
    "validation_mae": [
        mean_absolute_error(
            random_val_true, weight * random_schnet_val + (1 - weight) * random_lgbm_val
        )
        for weight in weights
    ],
})
best_weight = float(validation_curve.loc[validation_curve["validation_mae"].idxmin(), "schnet_weight"])
print(f"Validation-selected SchNet weight: {best_weight:.2f}")

random_ensemble = best_weight * random_schnet_test + (1 - best_weight) * random_lgbm_test
scaffold_ensemble = best_weight * scaffold_schnet_test + (1 - best_weight) * scaffold_lgbm_test
random_predictions["ensemble_prediction"] = random_ensemble
random_predictions["ensemble_absolute_error"] = np.abs(random_test_true - random_ensemble)
scaffold_predictions["ensemble_prediction"] = scaffold_ensemble
scaffold_predictions["ensemble_absolute_error"] = np.abs(scaffold_test_true - scaffold_ensemble)
random_predictions.to_csv(RESULTS_DIR / "final_predictions_random_test.csv", index=False)
scaffold_predictions.to_csv(RESULTS_DIR / "final_predictions_scaffold_test.csv", index=False)


ensemble_rows = [
    metric_row("random", f"Ensemble (w={best_weight:.2f})", random_test_true, random_ensemble,
               validation_curve["validation_mae"].min()),
    metric_row("scaffold", f"Ensemble (w={best_weight:.2f})", scaffold_test_true, scaffold_ensemble,
               mean_absolute_error(scaffold_val_true,
                   best_weight * scaffold_schnet_val + (1-best_weight) * scaffold_lgbm_val)),
]
final_comparison = pd.concat([base_rows.drop(columns=[
    "absolute_test_mae_difference_vs_lightgbm",
    "relative_test_mae_difference_vs_lightgbm_pct",
]), pd.DataFrame(ensemble_rows)], ignore_index=True)
final_comparison["absolute_test_mae_difference_vs_lightgbm"] = np.nan
final_comparison["relative_test_mae_difference_vs_lightgbm_pct"] = np.nan
for split_name in ["random", "scaffold"]:
    reference = final_comparison[
        (final_comparison["split"] == split_name)
        & final_comparison["model"].str.contains("LightGBM")
    ]["test_mae"].iloc[0]
    mask = final_comparison["split"].eq(split_name)
    final_comparison.loc[mask, "absolute_test_mae_difference_vs_lightgbm"] = (
        final_comparison.loc[mask, "test_mae"] - reference
    )
    final_comparison.loc[mask, "relative_test_mae_difference_vs_lightgbm_pct"] = (
        100 * final_comparison.loc[mask, "absolute_test_mae_difference_vs_lightgbm"] / reference
    )
final_comparison.to_csv(RESULTS_DIR / "final_model_comparison.csv", index=False)
display(final_comparison.sort_values(["split", "test_mae"]))

## 7. Presentation-ready figures

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
plot_rows = final_comparison.copy()
plot_rows["display_model"] = np.select([plot_rows["model"].str.contains("LightGBM"), plot_rows["model"].str.contains("Ensemble")], ["LightGBM", "Ensemble"], default="SchNet")
comparison_plot = plot_rows.pivot(index="display_model", columns="split", values="test_mae")
order = ["LightGBM", "SchNet", "Ensemble"]
x = np.arange(len(order)); width = .36
random_vals = comparison_plot.loc[order, "random"].to_numpy()
scaffold_vals = comparison_plot.loc[order, "scaffold"].to_numpy()
ax.bar(x - width/2, random_vals, width, label="Random test", color="#4C78A8")
ax.bar(x + width/2, scaffold_vals, width, label="Scaffold test", color="#F58518")
ax.set_xticks(x, order); ax.set_ylim(bottom=0)
ax.set_ylabel("MAE (stored units)"); ax.set_title("Model comparison by split")
ax.legend(frameon=False)
for xpos, value in zip(x - width/2, random_vals): ax.text(xpos, value, f"{value:.4f}", ha="center", va="bottom", fontsize=8)
for xpos, value in zip(x + width/2, scaffold_vals): ax.text(xpos, value, f"{value:.4f}", ha="center", va="bottom", fontsize=8)
save_figure("01_model_comparison_mae", FIGURE_DIR)

fig, axes = plt.subplots(2, 2, figsize=(10, 9))
for row, (split_name, frame) in enumerate([("Random", random_predictions), ("Scaffold", scaffold_predictions)]):
    target = frame["target"].to_numpy()
    prediction = frame["ensemble_prediction"].to_numpy()
    full_limits = [min(target.min(), prediction.min()), max(target.max(), prediction.max())]
    axes[row, 0].scatter(target, prediction, s=7, alpha=.25, color=MODEL_COLORS["Ensemble"], rasterized=True)
    axes[row, 0].plot(full_limits, full_limits, "--", color="black", linewidth=1)
    axes[row, 0].set(xlabel="True gap (stored units)", ylabel="Ensemble prediction (stored units)",
                     title=f"{split_name}: full range")
    limit = np.quantile(np.r_[target, prediction], .995)
    mask = (target <= limit) & (prediction <= limit)
    zoom_limits = [min(target[mask].min(), prediction[mask].min()), limit]
    axes[row, 1].hexbin(target[mask], prediction[mask], gridsize=40, mincnt=1, cmap="Greens")
    axes[row, 1].plot(zoom_limits, zoom_limits, "--", color="black", linewidth=1)
    axes[row, 1].set(xlim=zoom_limits, ylim=zoom_limits, aspect="equal",
                     xlabel="True gap (stored units)", ylabel="Ensemble prediction (stored units)",
                     title=f"{split_name}: central 99.5% (n={mask.sum():,})")
save_figure("02_best_final_predicted_vs_true", FIGURE_DIR)
display(Markdown("Each split retains a full-range panel. The paired density panel uses the split-specific 99.5th-percentile display limit only; ensemble metrics remain full-sample."))

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
frame = random_predictions
size_summary = frame.groupby("n_heavy_atoms")[["lightgbm_absolute_error", "schnet_absolute_error"]].median()
axes[0,0].plot(size_summary.index, size_summary["lightgbm_absolute_error"], marker="o", color=MODEL_COLORS["LightGBM"], label="LightGBM")
axes[0,0].plot(size_summary.index, size_summary["schnet_absolute_error"], marker="o", color=MODEL_COLORS["SchNet"], label="SchNet")
axes[0,0].set(xlabel="Heavy atoms", ylabel="Median absolute error", title="Random test: error vs size")
axes[0,0].legend(frameon=False)
frequency_group = pd.cut(frame["scaffold_train_frequency"], bins=[-1, 0, 1, 5, 20, np.inf], labels=["unseen", "singleton", "rare (2–5)", "medium (6–20)", "common (>20)"])
summary = frame.assign(scaffold_frequency_group=frequency_group).groupby("scaffold_frequency_group", observed=True)[["lightgbm_absolute_error", "schnet_absolute_error"]].median()
axes[0,1].plot(summary.index.astype(str), summary["lightgbm_absolute_error"], marker="o", color=MODEL_COLORS["LightGBM"], label="LightGBM")
axes[0,1].plot(summary.index.astype(str), summary["schnet_absolute_error"], marker="o", color=MODEL_COLORS["SchNet"], label="SchNet")
axes[0,1].set(xlabel="Scaffold-frequency group", ylabel="Median absolute error", title="Random test: scaffold familiarity")
axes[0,1].tick_params(axis="x", rotation=25)
limit = np.quantile(np.r_[frame["lightgbm_absolute_error"], frame["schnet_absolute_error"]], .995)
mask = (frame["lightgbm_absolute_error"] <= limit) & (frame["schnet_absolute_error"] <= limit)
axes[1,0].hexbin(frame.loc[mask, "lightgbm_absolute_error"], frame.loc[mask, "schnet_absolute_error"], gridsize=40, mincnt=1, cmap="viridis")
axes[1,0].plot([0, limit], [0, limit], "--", color="white", linewidth=1)
axes[1,0].set(xlim=(0, limit), ylim=(0, limit), aspect="equal", xlabel="LightGBM absolute error",
              ylabel="SchNet absolute error", title="Per-molecule errors: central 99.5%")
difference = frame["schnet_absolute_error"] - frame["lightgbm_absolute_error"]
diff_limit = difference.abs().quantile(.995)
axes[1,1].hist(difference[difference.abs() <= diff_limit], bins=50, color="#72B7B2", edgecolor="white")
axes[1,1].axvline(0, color="black", linestyle="--", linewidth=1)
axes[1,1].set(xlabel="SchNet error − LightGBM error", ylabel="Molecules", title="Paired error difference: central 99.5%")
save_figure("03_comparative_error_analysis", FIGURE_DIR)

plt.figure(figsize=(7, 4))
plt.plot(validation_curve["schnet_weight"], validation_curve["validation_mae"], color="#4C78A8")
plt.axvline(best_weight, color="black", linestyle="--", label=f"Selected w={best_weight:.2f}")
plt.xlabel("SchNet ensemble weight"); plt.ylabel("Random-validation MAE (stored units)")
plt.title("Validation-only ensemble selection"); plt.legend(frameon=False)
save_figure("04_ensemble_weight_validation", FIGURE_DIR)
display(Markdown("MAE comparisons and ensemble selection remain on linear axes. Density and paired-difference panels use 99.5% visual limits only; all reported metrics use complete test sets."))


## 8. Final conclusions

In [ ]:
random_scores = final_comparison.query("split == 'random'").set_index("model")["test_mae"]
scaffold_scores = final_comparison.query("split == 'scaffold'").set_index("model")["test_mae"]
ensemble_name = f"Ensemble (w={best_weight:.2f})"
random_winner = random_scores.idxmin()
scaffold_winner = scaffold_scores.idxmin()
random_ensemble_gain = min(
    random_scores["SchNet"], random_scores[classical_family]
) - random_scores[ensemble_name]
scaffold_ensemble_gain = min(
    scaffold_scores["SchNet"], scaffold_scores[classical_family]
) - scaffold_scores[ensemble_name]
random_residual_corr = residual_correlations.query("subset == 'random test'")["pearson"].iloc[0]
scaffold_residual_corr = residual_correlations.query("subset == 'scaffold test'")["pearson"].iloc[0]
random_rank_corr = residual_correlations.query("subset == 'random test'")["spearman"].iloc[0]
scaffold_rank_corr = residual_correlations.query("subset == 'scaffold test'")["spearman"].iloc[0]

delivery_model = (
    ensemble_name if random_ensemble_gain > 0 and scaffold_ensemble_gain >= 0
    else ("SchNet" if scaffold_scores["SchNet"] < scaffold_scores[classical_family]
          and random_scores["SchNet"] <= random_scores[classical_family] * 1.03
          else classical_family)
)

conclusion_text = f'''# Final conclusions

- **Best for interpolation:** `{random_winner}` has the lowest random-test MAE ({random_scores.min():.5f}).
- **Best for unseen scaffolds:** `{scaffold_winner}` has the lowest scaffold-test MAE ({scaffold_scores.min():.5f}).
- **Explicit geometry:** SchNet changes MAE by {random_scores[classical_family] - random_scores["SchNet"]:+.5f} on random test and {scaffold_scores[classical_family] - scaffold_scores["SchNet"]:+.5f} on scaffold test; its benefit is therefore concentrated where the positive difference is larger.
- **Error complementarity:** residual Pearson/Spearman correlations are {random_residual_corr:.3f}/{random_rank_corr:.3f} on random test and {scaffold_residual_corr:.3f}/{scaffold_rank_corr:.3f} on scaffold test. Pearson is dominated by shared extreme-target errors, while rank correlation better reflects typical-molecule complementarity.
- **Ensemble robustness:** validation selected `w={best_weight:.2f}` for SchNet. Relative to the better individual model, ensemble MAE improves by {random_ensemble_gain:+.5f} on random test and {scaffold_ensemble_gain:+.5f} on scaffold test.
- **Recommended delivery to Aqemia:** `{delivery_model}`. This recommendation prioritizes held-out MAE, scaffold robustness, and operational simplicity; the individual-molecule tables should guide follow-up inspection without causal overinterpretation.
'''
(RESULTS_DIR / "final_conclusions.md").write_text(conclusion_text, encoding="utf-8")
display(Markdown("## Final conclusions"))
display(Markdown(conclusion_text))

## Saved artifacts and interpretation

The notebook's scientific findings and conclusions are stated in the preceding sections. Its declared outputs are saved at the paths listed in the **Notebook contract** above. Implementation details shared across experiments live in `src/`; experiment choices and their interpretation remain visible here.
